In [ ]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter   # <-- changed this line

# Project paths
DATA_DIR = Path("data")
DB_DIR = Path("legal_db_v2")

print("Data directory:", DATA_DIR.resolve())
print("DB directory:", DB_DIR.resolve())
print("Files in data/:", list(DATA_DIR.glob("*.pdf")))

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
all_docs = []

for pdf_file in DATA_DIR.glob("*.pdf"):
    print(f"Loading {pdf_file.name} ...")
    loader = PyPDFLoader(str(pdf_file))
    pages = loader.load()
    
    # Tag each page with which act it came from (important for citations later)
    for page in pages:
        page.metadata["source_act"] = pdf_file.stem  # e.g. "ipc", "crpc", "rti", "consumer"
    
    all_docs.extend(pages)
    print(f"  → {len(pages)} pages loaded")

print(f"\nTotal pages loaded across all acts: {len(all_docs)}")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(all_docs)

print(f"Total chunks created: {len(chunks)}")
print("\nSample chunk:")
print("-" * 60)
print(chunks[0].page_content[:500])
print("-" * 60)
print("Metadata:", chunks[0].metadata)

In [ ]:
query = "What is the punishment for cheating under IPC?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} (source: {doc.metadata.get('source_act')}, page: {doc.metadata.get('page_label')}) ---")
    print(doc.page_content[:400])
    print()

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

In [ ]:
!pip uninstall google-generativeai -y
!pip install google-genai

In [ ]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Paths
DATA_DIR = Path("data")
DB_DIR = Path("legal_db_v2")

# Step 1: Load PDFs
all_docs = []
for pdf_file in DATA_DIR.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_file))
    pages = loader.load()
    for page in pages:
        page.metadata["source_act"] = pdf_file.stem
    all_docs.extend(pages)
print(f"Total pages loaded: {len(all_docs)}")

# Step 2: Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(all_docs)
print(f"Total chunks created: {len(chunks)}")

# Step 3: Build vector store
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=str(DB_DIR)
)

print(f"\n✅ Vector store built and saved to: {DB_DIR.resolve()}")
print(f"Total vectors stored: {vectorstore._collection.count()}")

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

In [ ]:
def detect_act_filter(query):
    query_lower = query.lower()
    if "rti" in query_lower or "right to information" in query_lower:
        return "rti"
    elif "consumer" in query_lower:
        return "consumer"
    elif "crpc" in query_lower or "criminal procedure" in query_lower:
        return "crpc"
    elif "ipc" in query_lower or "penal code" in query_lower:
        return "ipc"
    return None

def ask_legal_question(query, k=4):
    act_filter = detect_act_filter(query)

    if act_filter:
        results = vectorstore.similarity_search(query, k=k, filter={"source_act": act_filter})
    else:
        results = vectorstore.similarity_search(query, k=k)

    context_parts = []
    for doc in results:
        act = doc.metadata.get("source_act", "unknown").upper()
        page = doc.metadata.get("page_label", "?")
        context_parts.append(f"[Source: {act}, Page {page}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are a legal aid assistant helping Indian citizens understand their legal rights and options.
Answer the question using ONLY the context below. If the context doesn't fully answer the question, say so clearly.
Always mention which Act and Section your answer is based on. Keep the tone clear and non-intimidating for a layperson.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text, results

In [ ]:
!pip install python-dotenv

In [ ]:
query = "How do I file an RTI application?"
results = vectorstore.similarity_search(query, k=6)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} (source: {doc.metadata.get('source_act')}, page: {doc.metadata.get('page_label')}) ---")
    print(doc.page_content[:300])
    print()

In [ ]:
results = vectorstore.similarity_search(
    "How do I file an RTI application?", 
    k=4, 
    filter={"source_act": "rti"}
)
for i, doc in enumerate(results):
    print(f"--- Result {i+1} (page: {doc.metadata.get('page_label')}) ---")
    print(doc.page_content[:400])
    print()

In [ ]:
answer, sources = ask_legal_question("How do I file an RTI application?")
print("ANSWER:\n")
print(answer)
print("\n\nSOURCES USED:")
for doc in sources:
    print(f"- {doc.metadata.get('source_act', '?').upper()}, Page {doc.metadata.get('page_label', '?')}")

In [ ]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Paths
DATA_DIR = Path("data")
DB_DIR = Path("legal_db_v3")  # new folder, keeps v2 intact as backup

# Step 1: Load all PDFs (now includes bns.pdf and bnss.pdf)
all_docs = []
for pdf_file in DATA_DIR.glob("*.pdf"):
    print(f"Loading {pdf_file.name} ...")
    loader = PyPDFLoader(str(pdf_file))
    pages = loader.load()
    for page in pages:
        page.metadata["source_act"] = pdf_file.stem  # ipc, crpc, rti, consumer, bns, bnss
    all_docs.extend(pages)
    print(f"  → {len(pages)} pages loaded")

print(f"\nTotal pages loaded: {len(all_docs)}")

# Step 2: Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(all_docs)
print(f"Total chunks created: {len(chunks)}")

# Step 3: Build vector store
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=str(DB_DIR)
)

print(f"\n✅ Vector store built and saved to: {DB_DIR.resolve()}")
print(f"Total vectors stored: {vectorstore._collection.count()}")